# Global Rice Trade Network - Network Structure
Having cleaned the data, we now model the global rice trade as a **directed, weighted graph**, with:
- **Nodes** = countries
- **Edges** = rice explort flows (direction: exporter -> importer)
- **Edge Weight** = export volumes in tonnes

We then want to compute structural metrics to answer the research question:
- **Hubs** — which countries are the most influential exporters? (weighted
  out-degree, [PageRank](https://en.wikipedia.org/wiki/PageRank)
- **Chokepoints** — which countries sit on many trade paths? (betweenness
  centrality)
- **Vietnam's position** — where does Vietnam sit in the network?



In [2]:
import pandas as pd
import sqlite3
import networkx as nx

# Load cleaned trade data from the database
my_db = sqlite3.connect("data/rice.db")
trade = pd.read_sql("SELECT * FROM trade_clean", my_db)

print("Loaded trade_clean:", trade.shape)
print("\nYears available:", sorted(trade["Year"].unique()))
print("\nRows per year:")
print(trade.groupby("Year").size())

Loaded trade_clean: (37886, 4)

Years available: [np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Rows per year:
Year
2010    1874
2011    1953
2012    2324
2013    2418
2014    2415
2015    2518
2016    2656
2017    2769
2018    2631
2019    2670
2020    2774
2021    2775
2022    2754
2023    2831
2024    2524
dtype: int64


The number of trade flows (edges) varies by year:
- **2010-2011** have fewer flows -> A sparser network.
- **2020–2023** are the densest (~2,750–2,830 flows/year) — richer for analysis.

**We choose 2022 to build the structural network**:
- The data is dense (2,754 flows), so centrality metrics are reliable.
- This is recent enough to reflect the current structure, and precedes the 2023 India export ban. This aligns with the project design: comparing between the pre-shock network 2022 and 2023 to see how the network are restructured within the crisis. 

In [4]:
# Filter to a single year
year = 2022
trade_year = trade[trade["Year"] == 2022]

# Build a directed, weighted graph
graph = nx.from_pandas_edgelist(trade_year, source = "Reporter Countries", target = "Partner Countries", edge_attr = "Value", create_using = nx.DiGraph())

print(f"Graph for {year}:")
print("  Nodes (countries):", graph.number_of_nodes())
print("  Edges (trade flows):", graph.number_of_edges())

                                

Graph for 2022:
  Nodes (countries): 197
  Edges (trade flows): 2754


### Top 10 Exporting Hubs (2022) by Weighted out-degree
The weighted out-degree of a country is the total tonnage of rice it exported in 2022. 



In [12]:
# Weighted out-degree: Total rice exported by each country
out_deg_df = pd.DataFrame(out_deg.items(), columns=["Country", "Total Export"])
out_deg_df = out_deg_df.sort_values("Total Export", ascending=False)
out_deg_df = out_deg_df.head(10)   
out_deg_df# SQL: top exporters in 2022 only (same data the graph was built from)


                    Country  TotalExport
0                     India  17865534.21
1                  Thailand   6783499.86
2                  Viet Nam   5927194.42
3                  Pakistan   3121043.00
4           China, mainland   1903921.46
5  United States of America   1288302.88
6                     Italy    639562.16
7                  Cambodia    629254.05
8                   Uruguay    526979.13
9                  Paraguay    349885.87


**Observations:**
- The 2022 top exporters closely match the 15-year cumulative ranking (only **Paraguay** and **Brazil** swapping positions). This shows the export hierarchy is stable over time — the major hubs are structurally entrenched — rather than being a one-year artifact. Additionally, this close match also reflects the stability in the rice trade. 

### Exporting Hubs in 2022 using PageRank
While weighted out-degree measures **how much** a country exports, **PageRank** measures **the influence within the network's structure**. Specifically, a country ranks high if it exports to other well-connected countries, not just by the raw volume. 

Comparing the two different rankings shows whether the largest exporters are also the most structurally influential.


In [16]:
# PageRank: Structural influence in the network
graph_rev = graph.reverse()
pagerank = dict(nx.pagerank(graph_rev, weight = "Value"))

pagerank_df = pd.DataFrame(pagerank.items(), columns=["Country", "PageRank"]).sort_values("PageRank", ascending=False).head(10)
pagerank_df


,Country,PageRank
49,India,0.226497
109,Thailand,0.097444
57,United Arab Emirates,0.094404
116,Viet Nam,0.078775
54,Pakistan,0.074468
53,Oman,0.057870
2,Brazil,0.039425
29,Italy,0.021927
100,Paraguay,0.020330
10,United States of America,0.019652


**Observations:**
- **India** dominates even more under PageRank (0.226, which is about **twice** Thailand). This indicates that **India** does not only exports the most, but also exports to well-connected countries (that exports to other countries) - its systemic influence exceeds what raw volume suggests.
- **UAE** and **Oman** rank high on PageRank despite modest export volume (as shown in the Weighted out-degree ranking). Their influence probably comes from being **re-export hubs** (importing from one or many countries, then re-exporting).
- This shows that PageRank captures influence that weighted out-degree misses, and flags UAE and Oman as candidates for the chokepoint (betweenness). 
